# AIMarx TRAIN-05 — guarded Qwen2.5-3B QLoRA feasibility

Free-tier Tesla T4 only. This is a new model baseline, not a resume of Qwen3-0.6B. Run through the one-step checkpoint, inspect it, then explicitly authorize the five-step resume. No Drive mount, Hub push, paid compute, smoke-test split, or 20-step pilot.


In [ ]:
import os, pathlib, subprocess, sys
assert os.path.exists('/content'), 'Run in Google Colab'
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
REPO = 'https://github.com/hongkhang21998-creator/AIMarx.git'
PINNED_COMMIT = '2b17afdd1bc31c5100bb89ccd3d1f9e66dd2e1e0'
subprocess.run(['git', 'clone', REPO, '/content/AIMarx'], check=True)
subprocess.run(['git', '-C', '/content/AIMarx', 'checkout', '--detach', PINNED_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', '/content/AIMarx', 'rev-parse', 'HEAD'], text=True).strip() == PINNED_COMMIT
os.chdir('/content/AIMarx')


## Install pinned QLoRA dependencies and verify inputs


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/colab_qwen25_3b/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.prepare', '/content/aimarx-qwen25-3b-data'], check=True)


## Gate 1: train exactly one optimizer step

This cell may download the pinned model. It fails closed for insufficient GPU/disk, over-length records, OOM, or an incomplete checkpoint.


In [ ]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.train', '--data', '/content/aimarx-qwen25-3b-data', '--output', '/content/aimarx-qwen25-3b-output', '--stop-after', '1'], check=True)
checkpoint_1 = pathlib.Path('/content/aimarx-qwen25-3b-output/checkpoint-1')
assert checkpoint_1.exists()
print((checkpoint_1 / 'aimarx-manifest.json').read_text())


## Human gate: inspect step 1 before resume

Confirm there was no OOM, the manifest is complete, and T4 resources remain healthy. Change the flag to `True` only after review.


In [ ]:
APPROVE_RESUME_TO_5 = False
assert APPROVE_RESUME_TO_5 is True, 'STOP: human approval required before step 1 -> 5'


In [ ]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.train', '--data', '/content/aimarx-qwen25-3b-data', '--output', '/content/aimarx-qwen25-3b-output', '--resume-from', str(checkpoint_1)], check=True)
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.evaluate', '--data', '/content/aimarx-qwen25-3b-data', '--checkpoint', '/content/aimarx-qwen25-3b-output/checkpoint-5', '--output', '/content/aimarx-qwen25-3b-output/evaluation.json'], check=True)


In [ ]:
import hashlib, shutil
archive = pathlib.Path(shutil.make_archive('/content/AIMarx-Qwen2.5-3B-QLoRA-step5', 'zip', '/content/aimarx-qwen25-3b-output'))
print({'bytes': archive.stat().st_size, 'sha256': hashlib.sha256(archive.read_bytes()).hexdigest()})
from google.colab import files
files.download(str(archive))
